# 06 Robustness Checks (MVP 3.3)

This notebook implements **Placebo Test (Permutation Test)** to validate that the estimated treatment effect disappears when the treatment assignment is randomized.

Outputs:
- `outputs/figures/fig_09_placebo_ate.png`
- `data/processed/placebo_results.json`


## Section 0: Setup

Conventions (consistent with notebooks 03-05):
- 检测 `project_root`, 并添加到 `sys.path`, 通过 `os.chdir(project_root)` 更换到项目目录
- 加载 `configs/config.yml` 配置
- 使用 config 中的路径 (保证代码的可迁移性) 以及协变量


In [1]:
# ======================================================
# Section 0 (Cell 1/2): Imports + Project Root + Config
# ======================================================
# NOTE: 
# We implement an in-notebook matching helper to avoid repeated disk writes during placebo.
# No new src functions).

import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from sklearn.neighbors import NearestNeighbors

# project_root detection (robust to running from ./notebooks)
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'configs' / 'config.yml').exists() else cwd.parent
if not (project_root / 'configs' / 'config.yml').exists():
    raise FileNotFoundError('Cannot find configs/config.yml from current working directory')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

# Load config
config_candidates = [
    project_root / 'configs' / 'config.yml',
    project_root / 'configs' / 'config.yaml',
]
config_path = next((p for p in config_candidates if p.exists()), None)
if config_path is None:
    raise FileNotFoundError('Missing config.yml/config.yaml under configs/')
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# time anchor (audit-friendly)
tz_utc8 = timezone(timedelta(hours=8))
print('[Time]', datetime.now(tz_utc8).strftime('%Y-%m-%d %H:%M:%S %Z%z'))

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Paths
figures_dir = Path(config['paths']['figures_dir'])
figures_dir.mkdir(parents=True, exist_ok=True)

processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

# Columns
covariates = list(config['data']['covariates'])
treatment_col = str(config['data']['treatment_col'])
outcome_col = str(config['data']['outcome_col'])
random_state = int(config.get('general', {}).get('random_state', 42))

print('[Cols] treatment:', treatment_col, '| outcome:', outcome_col)
print('[Covariates]', len(covariates))
print('[Seed]', random_state)

from src.causal import estimate_ps, compute_ate  # imported after sys.path setup


[Time] 2026-03-02 22:23:16 UTC+08:00+0800
[Cols] treatment: treatment | outcome: conversion
[Covariates] 9
[Seed] 42


In [2]:
# ======================================================
# Section 0 (Cell 2/2): Load Data + Baseline ATE
# ======================================================

features_path = Path(config['paths']['features_data'])
if not features_path.exists():
    raise FileNotFoundError(f'Missing features_data: {features_path}')

df = pd.read_csv(features_path)
print('[Loaded]', features_path, '| shape:', df.shape)

missing_covs = [c for c in covariates if c not in df.columns]
if missing_covs:
    raise ValueError(f'Missing covariates in features df: {missing_covs}')

X = df[covariates].copy()
T = pd.to_numeric(df[treatment_col], errors='coerce').astype(int)
Y = pd.to_numeric(df[outcome_col], errors='coerce').astype(int)

# Row id for mapping matched rows back to covariates (balance check on original match only).
row_id_col = '__row_id'
row_ids = np.arange(len(df), dtype=int)

# Baseline ATE reference (naive, RCT): used as a sanity check only
ate_naive = float(Y[T == 1].mean() - Y[T == 0].mean())
print(f'[Naive ATE] {ate_naive:.6f} ({ate_naive:.3%})')

# Baseline ATE reference (PSM): run once to get a comparable original_ate + CI
ps, _ = estimate_ps(X, T, random_state=random_state)
# Keep a minimal frame for matching/ATE to avoid full DataFrame copies in loops.
df_with_ps = pd.DataFrame({
    row_id_col: row_ids,
    outcome_col: Y.to_numpy(copy=False),
    treatment_col: T.to_numpy(copy=False),
    'ps': np.asarray(ps, dtype=float),
})


[Loaded] data\processed\hillstrom_features.csv | shape: (64000, 16)
[Naive ATE] 0.004955 (0.495%)


## Section 1: Placebo Test (Permutation Test)

对处理分配 `T` 进行置换，并重新进行 **倾向性评分估计（PS estimation）+ 匹配 + ATE** 的执行链路
在零假设（无因果效应）下，经过 Placebo 处理后的 ATE 分布应集中在0附近

In [3]:
# ======================================================
# Section 1 (Cell 1/4): Helpers (in-notebook)
# ======================================================

def _match_ps_in_memory(
    df_in: pd.DataFrame,
    *,
    ps_col: str = 'ps',
    treatment_col: str = 'treatment',
    caliper_factor: float = 0.2,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    In-memory 1:1 nearest-neighbor matching on propensity score (no replacement).

    Why in-notebook?
    - `src.causal.match_ps()` 有 "固定将数据落盘写成 CSV" 的副作用, Placebo 是一次性验证, 应该 notebook 内联, 避免过度工程化
    - 保持算法参数 (NN + caliper + no replacement) 与 uplift 基线的相同, 并且不产生副作用
    """
    if ps_col not in df_in.columns:
        raise ValueError(f'Missing ps_col: {ps_col}')
    if treatment_col not in df_in.columns:
        raise ValueError(f'Missing treatment_col: {treatment_col}')

    # Treat df_in as read-only; only matched slices are copied later.
    df_local = df_in

    ps_series = pd.to_numeric(df_local[ps_col], errors='coerce')
    if ps_series.isnull().any():
        raise ValueError('ps contains NaN/non-numeric values')
    ps = ps_series.astype(float).to_numpy()
    if np.isnan(ps).any() or (not np.isfinite(ps).all()):
        raise ValueError('ps contains NaN/inf')

    t_series = pd.to_numeric(df_local[treatment_col], errors='coerce')
    if t_series.isnull().any():
        raise ValueError('treatment contains NaN/non-numeric values')
    t = t_series.astype(int).to_numpy()
    if not set(np.unique(t)).issubset({0, 1}):
        raise ValueError('treatment must be binary (0/1)')

    treated_idx = np.where(t == 1)[0]
    control_idx = np.where(t == 0)[0]
    if treated_idx.size == 0 or control_idx.size == 0:
        raise ValueError('Both treated and control must be non-empty')

    ps_std = float(np.std(ps))

    # Special case (important for placebo):
    #   1. 如果 PS 几乎为常数， kNN 相邻匹配列表可能会相同 (所有对照组用户与处理组用户的 "distance" 都一样)
    #   2. 此时贪婪算法的不放回匹配可能会失败 (重复选择同一个对照组用户)。
    #   3. 在此情况下，采用随机的 1:1 配对是有效的 (从复杂的匹配算法等价退化为随机配对)
    #   4. 空值匹配基线（基于常数 PS 的匹配不包含任何信息）。
    if ps_std < 1e-4:
        rng = np.random.RandomState(random_state)
        n_pairs = int(min(treated_idx.size, control_idx.size))

        # Matched pairs check
        if n_pairs < 10:
            raise ValueError(f'Too few matched pairs: {n_pairs}')
        
        # rng.choice(..., replace=False): No-replacement exact sampling (N = n_pairs)
        chosen_ctrl = rng.choice(control_idx, size=n_pairs, replace=False)
        chosen_treat = rng.choice(treated_idx, size=n_pairs, replace=False)

        # Generate match_id
        pair_ids = np.arange(n_pairs, dtype=int)
        df_treat = df_local.iloc[chosen_treat].copy()
        df_ctrl = df_local.iloc[chosen_ctrl].copy()
        df_treat['match_id'] = pair_ids
        df_ctrl['match_id'] = pair_ids
        return pd.concat([df_treat, df_ctrl], ignore_index=True)

    caliper = float(caliper_factor) * ps_std
    if not np.isfinite(caliper) or caliper < 0:
        raise ValueError('Invalid caliper computed from ps')
    caliper = max(float(caliper), 1e-6)

    # Efficiency note:
    # - 控制组是 Hillstrom 中的较小组 (T:C ~ 2:1)。
    # - 根据已有的 PS 对控制组进行 NN 查询 (反向匹配)，以提升效率
    # (让数量较少的组 (control) 去匹配较大的组 (treatment), 减少时间复杂度)
    treated_ps = ps[treated_idx].reshape(-1, 1)
    control_ps = ps[control_idx].reshape(-1, 1)

    desired_pairs = int(min(treated_idx.size, control_idx.size))

    # Try a smaller k first for speed; fall back to larger k if matching is poor.
    distances = None
    neighbors = None
    matched_ctrl = []
    matched_treat = []

    for k_try in [50, 200]:
        k = int(min(k_try, treated_idx.size))
        k = max(1, k)
        nn = NearestNeighbors(n_neighbors=k, algorithm='kd_tree')
        nn.fit(treated_ps)
        distances, neighbors = nn.kneighbors(control_ps, return_distance=True)

        ctrl_order = np.argsort(distances[:, 0], kind='mergesort')
        used_treated = set()
        matched_ctrl = []
        matched_treat = []

        for ctrl_pos in ctrl_order:
            # If nearest neighbor is farther than caliper, skip this crtl_pos
            if float(distances[ctrl_pos, 0]) > caliper:
                continue

            chosen_treat_pos = None
            for j in range(k):
                if float(distances[ctrl_pos, j]) > caliper:
                    break
                treat_pos = int(neighbors[ctrl_pos, j])
                if treat_pos not in used_treated:
                    chosen_treat_pos = treat_pos
                    break
            # If still None:means no suitable treatment found, skip this ctrl_pos
            if chosen_treat_pos is None:
                continue
            used_treated.add(int(chosen_treat_pos))
            matched_ctrl.append(int(control_idx[int(ctrl_pos)]))
            matched_treat.append(int(treated_idx[int(chosen_treat_pos)]))

        match_rate = float(len(matched_ctrl)) / float(desired_pairs)
        if match_rate >= 0.90 or k == treated_idx.size:
            break

    # (matching already computed in the loop above)
    n_pairs = int(len(matched_ctrl))
    if n_pairs < 10:
        raise ValueError(f'Too few matched pairs: {n_pairs}')

    # Build matched_df in one go (avoid per-row copies).
    pair_ids = np.arange(n_pairs, dtype=int)
    df_treat = df_local.iloc[matched_treat].copy()
    df_ctrl = df_local.iloc[matched_ctrl].copy()
    df_treat['match_id'] = pair_ids
    df_ctrl['match_id'] = pair_ids
    out = pd.concat([df_treat, df_ctrl], ignore_index=True)
    return out


def _ate_point_estimate_from_pairs(
    matched_df: pd.DataFrame,
    *,
    outcome_col: str,
    treatment_col: str,
) -> float:
    """Point estimate ATE using matched pairs (treated - control per match_id)."""
    if 'match_id' not in matched_df.columns:
        raise ValueError('matched_df must contain match_id')

    t = pd.to_numeric(matched_df[treatment_col], errors='coerce')
    y = pd.to_numeric(matched_df[outcome_col], errors='coerce')
    if t.isnull().any() or y.isnull().any():
        raise ValueError('treatment/outcome contains NaN')

    pair_check = (
        matched_df.assign(_t=t.astype(int))
        .groupby('match_id', dropna=False)['_t']
        .agg(size='size', treated_sum='sum')
    )
    if (pair_check['size'] != 2).any() or (pair_check['treated_sum'] != 1).any():
        raise ValueError('Invalid pairing: each match_id must have exactly 2 rows (1 treated, 1 control)')

    pair_outcomes = (
        matched_df.assign(_t=t.astype(int), _y=y.to_numpy(dtype=float, copy=False))
        .pivot_table(index='match_id', columns='_t', values='_y', aggfunc='first')
    )
    diffs = (pair_outcomes[1] - pair_outcomes[0]).to_numpy(dtype=float, copy=False)
    return float(np.mean(diffs))


def _ps_smd(ps_arr: np.ndarray, t_arr: np.ndarray, eps: float = 1e-12) -> float:
    """Cheap balance signal: SMD on propensity score (raw ps)."""
    ps_vec = np.asarray(ps_arr, dtype=float).reshape(-1)
    t_vec = np.asarray(t_arr, dtype=int).reshape(-1)
    if ps_vec.size == 0:
        raise ValueError('ps_arr cannot be empty')
    if ps_vec.size != t_vec.size:
        raise ValueError('ps_arr and t_arr length mismatch')
    if not np.isfinite(ps_vec).all():
        raise ValueError('ps_arr contains NaN/inf')
    if not set(np.unique(t_vec)).issubset({0, 1}):
        raise ValueError('t_arr must be binary (0/1)')

    ps_t = ps_vec[t_vec == 1]
    ps_c = ps_vec[t_vec == 0]
    if ps_t.size == 0 or ps_c.size == 0:
        raise ValueError('Both treated and control must be non-empty')

    denom = float(np.std(ps_vec))
    denom = max(denom, float(eps))
    return float(abs(float(ps_t.mean()) - float(ps_c.mean())) / denom)


def _quick_match_metrics(
    matched_df: pd.DataFrame,
    *,
    desired_pairs: int,
    n_treated_total: int,
    n_control_total: int,
    treatment_col: str,
    ps_col: str = 'ps',
) -> dict:
    """Loop-safe matching metrics: coverage + cheap balance (PS SMD)."""
    if matched_df is None or matched_df.empty:
        raise ValueError('matched_df cannot be empty')
    if 'match_id' not in matched_df.columns:
        raise ValueError('matched_df must contain match_id')

    # Number of matched pairs
    n_pairs = int(matched_df['match_id'].nunique())
    # Theoretical maximum match rate = n_pairs / min{n_treated, n_control}
    match_rate_max = float(n_pairs) / float(max(1, desired_pairs))
    # Used ratio of control samples
    match_rate_control = float(n_pairs) / float(max(1, n_control_total))
    # Used ratio of treated samples
    treated_utilization = float(n_pairs) / float(max(1, n_treated_total))

    t_vec = pd.to_numeric(matched_df[treatment_col], errors='coerce')
    if t_vec.isnull().any():
        raise ValueError('treatment contains NaN/non-numeric values')
    t_arr = t_vec.astype(int).to_numpy()

    ps_vec = pd.to_numeric(matched_df[ps_col], errors='coerce')
    if ps_vec.isnull().any():
        raise ValueError('ps contains NaN/non-numeric values')
    ps_arr = ps_vec.astype(float).to_numpy()

    # SMD of PS after matching
    ps_smd_after = _ps_smd(ps_arr, t_arr)

    treated_n = int((t_arr == 1).sum())
    control_n = int((t_arr == 0).sum())

    # Check integrity of matched_df:
    integrity_counts_ok = (treated_n == n_pairs) and (control_n == n_pairs) and (len(matched_df) == 2 * n_pairs)

    return {
        'n_pairs': int(n_pairs),
        'match_rate_max': float(match_rate_max), 
        'match_rate_control': float(match_rate_control),
        'treated_utilization': float(treated_utilization),
        'ps_smd_after': float(ps_smd_after),
        'integrity_counts_ok': bool(integrity_counts_ok),
    }


**Helper: _match_ps_in_memory()**:
1) **ALgorithm**: 贪婪不放回匹配算法,对于每个 control ，选一个不冲突的且使得总距离小的 treatment 进行匹配。如果后续需要使用全局最优算法 (性能开销大), 可以采用 Assignment Problem (Hungarian Algorithm)
    - 建立代价矩阵 $C_{ij} = D_{ij}$, 把超出 caliper 的边设为无穷大
    - 求解 $min(C_{ij})$ 的最优解 (做最小代价匹配--Hungarian / min cost flow)
2) **Numerical processing**: 
    - 代码使用的是 $distance = |ps_c - ps_t|$ 和 $caliper = 0.2 * std(ps)$
    - 文献中常用 $distance = |logit(ps_c) - logit(ps_t)|$ 和 $caliper = 0.2 * sd(logit(ps))$(Logistic 变换) 
    - 本项目为了保持与 Phase 2 主链路一致性, 默认使用前者。后者作为可选拓展用语进一步稳健性对照

**Logistic 变换**: $\text{logit}(p) = \ln\left(\frac{p}{1-p}\right)$

**Helper: _ate_point_estimate_from_pairs()**:
1) **ATE Computation**:
  - 公式：$$\widehat{ATE}_{pair}=\frac{1}{n}\sum_{k=1}^{n}\left(Y_{k,1}-Y_{k,0}\right)$$
  - 方法: 在 PSM 后用 "配对差分均值" 估计处理效应：对每个匹配对计算 Y_treated - Y_control，再对所有匹配对取平均。
  - 估计对象的变化: 由于 caliper/no-replacement 会丢弃不可匹配样本，这个估计量更接近 overlap/matched sample 上的平均处理效应 (ATT on matched sample)
  - 优势: Matching 的价值在于降低可观测混杂后, 配对差值更稳定 (方差更小);
  - 预见性防御: 由于配对过程中会丢弃样本, 如果样本量损失过大会导致不稳定。因为代码中对匹配率进行了监控